# 5. Composition (horizon H2) : stabilité du onze, qualité du XI, entraîneur

**Question** : à quoi ressemblent les variables de la composition (groupes G4, G5, G6), sur quelle part des lignes existent-elles, comment bougent-elles au fil de la saison, et en quoi se recoupent-elles entre elles et avec la force des équipes ?

> Notebook **descriptif** (partie 6, lot 6.6, ADR-0034 et ADR-0047). Il ne sert ni à choisir une variable ni à régler un paramètre :
> la demi-vie de la cohésion se choisit par validation interne du pli, les variantes de stabilité sont mesurées en diagnostic
> et les groupes s'ajoutent dans l'ordre G6, G5, G4 (6.7). **Corrélation n'est pas apport.**
> Données : jeu H2 `ds-2026-11-01-94e07897`, période de développement seulement (matchs antérieurs au 1er juillet 2025,
> scellé de l'ADR-0012), lu par la porte `foot_predictor.features.sources`. Sorties effacées au commit (`nbstripout`) : réexécuter pour les voir.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Seule porte vers les données (ADR-0028) : jeu versionné H2 (G0 à G6, 92 colonnes).
from foot_predictor.features.sources import load_dataset

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
VERSION = "ds-2026-11-01-94e07897"  # jeu H2 de 6.5 ; le jeu le plus récent peut changer
frame, manifest = load_dataset(VERSION)
print("Jeu de données :", manifest["version"], "-", len(frame), "lignes, horizon", manifest["scope"]["horizon"])
NAMES = {
    39: "PL",
    40: "Champ.",
    140: "Liga",
    141: "Segunda",
    78: "BL",
    79: "2. BL",
    135: "Serie A",
    136: "Serie B",
    61: "L1",
    62: "L2",
}
COHESION = [f"xi_cohesion_ewm_h{h}" for h in ("3", "5", "10", "20", "inf")]
G6 = ["xi_overlap_prev", *COHESION, "xi_new_starters"]
G5 = ["xi_quality", "xi_regulars_missing", "xi_mean_age"]
G4 = ["coach_tenure_days", "coach_is_new"]
ALL = G4 + G5 + G6  # colonnes de l'équipe (les colonnes « opp_ » sont symétriques)
top5 = frame[frame["eval_population"]].copy()
top5["saison"] = top5["season_year"].astype(str) + "-" + ((top5["season_year"] + 1) % 100).astype(str).str.zfill(2)
top5["journee"] = pd.to_numeric(top5["round"].str.extract(r"(\d+)\s*$")[0], errors="coerce")
# Période étudiée : 2016-17 à 2024-25 (2015-16 sert de rodage à G5, 2016-17 à G4 ; D3, ADR-0047).
study = top5[top5["season_year"].between(2016, 2024)].copy()
print("Top 5 :", len(top5), "lignes ; étudiées (2016-17 à 2024-25) :", len(study))

## Couverture : part de valeurs vides (%) par saison, top 5

In [ ]:
cover = top5.groupby("saison")[
    ["coach_tenure_days", "xi_quality", "xi_regulars_missing", "xi_mean_age", "xi_overlap_prev", "xi_cohesion_ewm_h5"]
].agg(lambda s: s.isna().mean() * 100)
display(cover.loc["2015-16":].round(2))

## Distributions (top 5, 2016-17 à 2024-25, colonnes de l'équipe)

In [ ]:
display(study[ALL].describe().T[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]].round(3))
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax, (column, bins) in zip(
    axes.ravel(),
    [
        ("xi_overlap_prev", np.arange(-0.5, 12)),
        ("xi_cohesion_ewm_h5", 40),
        ("xi_quality", 40),
        ("xi_mean_age", 40),
        ("xi_regulars_missing", 30),
        ("coach_tenure_days", 40),
    ],
):
    ax.hist(study[column].dropna(), bins=bins, color="#4c78a8")
    ax.set_title(column)
fig.tight_layout()
plt.show()

## Début de saison (rapport H.4) : les variables selon la journée

In [ ]:
study["tranche"] = pd.cut(study["journee"], [0, 5, 10, 19, 60], labels=["1-5", "6-10", "11-19", "20 et plus"])
rows = []
for name in [
    "xi_overlap_prev",
    "xi_cohesion_ewm_h3",
    "xi_cohesion_ewm_h10",
    "xi_cohesion_ewm_hinf",
    "xi_new_starters",
    "xi_regulars_missing",
    "xi_quality",
    "coach_is_new",
]:
    g = study.groupby("tranche", observed=True)[name]
    rows.append(pd.concat([g.mean().rename("moyenne"), g.std().rename("écart type")], axis=1).assign(variable=name))
display(
    pd.concat(rows).reset_index().pivot(index="variable", columns="tranche", values=["moyenne", "écart type"]).round(3)
)
by_round = study.groupby("journee")[["xi_overlap_prev", "xi_cohesion_ewm_h5", "xi_cohesion_ewm_hinf"]].mean()
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
by_round["xi_overlap_prev"].plot(ax=axes[0], title="Titulaires communs avec le match précédent (moyenne par journée)")
by_round[["xi_cohesion_ewm_h5", "xi_cohesion_ewm_hinf"]].plot(ax=axes[1], title="Cohésion (moyenne par journée)")
fig.tight_layout()
plt.show()

## Corrélations entre variantes de stabilité (rapport H.3 et H.8)

In [ ]:
display(study[G6 + ["xi_regulars_missing"]].corr().round(2))

## Composition et force des équipes : confusion possible

Corrélation de chaque variable avec l'Elo (la force) et avec les buts marqués, puis avec les buts **à Elo donné** (les deux corrélés après retrait de la part linéaire expliquée par l'Elo). Un lien brut peut n'être que la force.

In [ ]:
sub = study.dropna(subset=["elo_pre"])


def residual(y, x):
    ok = y.notna() & x.notna()
    slope, intercept = np.polyfit(x[ok], y[ok], 1)
    return y - (slope * x + intercept)


goals_res = residual(sub["goals_for"].astype(float), sub["elo_pre"])
rows = []
for name in ALL:
    v = sub[name].astype(float)
    rows.append(
        {
            "variable": name,
            "avec Elo": v.corr(sub["elo_pre"]),
            "avec buts marqués": v.corr(sub["goals_for"].astype(float)),
            "avec buts, à Elo donné": residual(v, sub["elo_pre"]).corr(goals_res),
        }
    )
display(pd.DataFrame(rows).set_index("variable").round(3))

## Qualité du XI : comparabilité entre championnats et dérive dans le temps

In [ ]:
q = frame[frame["season_year"].between(2016, 2024)].copy()
q["championnat"] = q["api_league_id"].map(NAMES)
display(q.groupby("championnat")[["xi_quality", "xi_mean_age", "xi_regulars_missing"]].mean().round(3).T)
display(study.groupby("saison")[["xi_quality", "xi_mean_age"]].agg(["mean", "std"]).round(3))

## Entraîneur : ancienneté plafonnée, nouvel entraîneur, censure à gauche

In [ ]:
study["anciennete"] = pd.cut(
    study["coach_tenure_days"],
    [-1, 0, 30, 90, 180, 364, 365],
    labels=["0 (arrivée)", "1-30", "31-90", "91-180", "181-364", "365 (plafond)"],
)
display(
    study.groupby("anciennete", observed=True)
    .agg(
        lignes=("goals_for", "size"),
        buts_pour=("goals_for", "mean"),
        buts_contre=("goals_against", "mean"),
        elo=("elo_pre", "mean"),
    )
    .round(3)
)
display(
    study.groupby("coach_is_new")
    .agg(
        lignes=("goals_for", "size"),
        buts_pour=("goals_for", "mean"),
        buts_contre=("goals_against", "mean"),
        elo=("elo_pre", "mean"),
    )
    .round(3)
)
cens = (
    top5[top5["season_year"].between(2016, 2019)]
    .groupby("saison")
    .agg(
        nouveaux=("coach_is_new", lambda s: (s == 1).mean() * 100),
        plafonnes=("coach_tenure_days", lambda s: (s == 365).mean() * 100),
    )
)
display(cens.round(1))

## Conclusion

Chiffres de l'exécution du 2026-11-01 sur `ds-2026-11-01-94e07897` (top 5, 2016-17 à 2024-25, sauf mention).

- **Couverture** : à partir de 2021-22, au plus 0,3 % de valeurs vides pour les variables de la composition (0,0 % en 2024-25). Le rodage se voit : en 2015-16, G4 est vide à 100 % (pas d'entraîneur) et `xi_quality` à 1,0 % ; en 2016-17, `xi_mean_age` à 3,5 % et `xi_quality` à 1,2 %.
- **Distributions** : recouvrement moyen de 8,3 titulaires sur 11 (écart type 1,7) ; cohésion h5 de 0,41 (0,12) ; âge moyen de 27,0 ans (1,4). `xi_quality` est très resserrée (6,96, écart type 0,09) : les écarts entre équipes sont petits devant la note elle-même. L'ancienneté de l'entraîneur est plafonnée à 365 jours pour 45 % des lignes ; 11 % des lignes ont un nouvel entraîneur.
- **Début de saison (rapport H.4)** : l'intuition se vérifie en partie. Aux journées 1 à 5, la variance du recouvrement est plus forte (écart type 2,0 contre 1,5 à 1,7 ensuite), il y a 2,3 nouveaux titulaires contre 0,5 après la journée 19, 42 % d'habituels absents contre 29 à 31 %, et 36 % de nouveaux entraîneurs contre 5 à 10 %. Les cohésions courtes montent avec la journée (h3 : 0,37 puis 0,47 ; h10 : 0,28 puis 0,36), alors que la cohésion à historique intégral (`hinf`) ne bouge presque pas (0,058 puis 0,068, écart type 0,05) : sans décroissance, elle est inerte, comme l'annonçait H.4. `xi_quality` ne dépend pas de la journée.
- **Variantes de stabilité (H.3)** : les cohésions voisines sont très corrélées (h3 et h5 : 0,98 ; h5 et h10 : 0,96 ; h10 et h20 : 0,96), les éloignées moins (h3 et h20 : 0,76). La corrélation avec le recouvrement du dernier match baisse de 0,71 (h3) à 0,11 (hinf) : **l'annonce de plus de 0,9 entre paires et recouvrement ne se vérifie pas** sans remise à zéro, sauf entre demi-vies voisines. Les habituels absents recoupent fortement la cohésion (−0,85 avec h5, −0,83 avec h3) : G5 et G6 ne sont pas indépendants sur ce point, et c'est à l'ablation de 6.7 (G5 ajouté après G6) de dire s'il reste quelque chose.
- **Confusion avec la force** : `xi_quality` est corrélée à 0,83 avec l'Elo, c'est presque un second indicateur de force. Corrélation avec les buts marqués : 0,28 (contre 0,33 pour l'Elo, notebook 2). À Elo donné (part linéaire retirée des deux côtés), elle tombe à 0,07 ; toutes les autres variables sont à moins de 0,02 en valeur absolue (cohésion : 0,00 à 0,01 ; ancienneté : −0,01). Les liens bruts de la cohésion longue avec les buts (0,07 pour h20) viennent de la force des équipes (corrélation de 0,25 avec l'Elo). Lecture : le gain attendu de H2 est petit, comme le rapport et l'ADR-0047 le disaient d'avance ; seule la mesure en plis, à variables données, tranchera.
- **Qualité du XI entre championnats (D9)** : les notes moyennes vont de 6,88 (Segunda) à 6,99 (Premier League). Dans chaque pays, la D2 est en dessous de la première division (Championship 6,90 contre 6,99 ; Segunda 6,88 contre 6,94 ; 2. Bundesliga 6,92 contre 6,95 ; Serie B 6,91 contre 6,95 ; Ligue 2 6,90 contre 6,95). Pas de dérive d'une saison à l'autre (6,95 à 6,96). L'âge moyen est plus élevé en Espagne et en Italie (27,1 à 27,6 ans) qu'en France, en Allemagne et en D2 anglaise (26,5 à 26,7).
- **Entraîneur** : l'ancienneté et les buts vont de pair (1,35 but marqué à l'arrivée, 1,43 au plafond), mais l'Elo aussi (1 507 contre 1 562) ; de même pour « nouvel entraîneur » (1,31 contre 1,41 but, Elo 1 509 contre 1 548). Censure à gauche visible en 2016-17 : 19,5 % de nouveaux entraîneurs (10 % ensuite) et 0 % de lignes au plafond (47 à 52 % ensuite), d'où l'apprentissage à partir de 2017-18 (D3).

**À retenir pour 6.7** : aucune variable ni aucun paramètre n'est choisi ici. Les points à surveiller sont la redondance entre les habituels absents et la cohésion, la quasi-identité de `xi_quality` avec l'Elo, et l'inertie de `hinf`.